In [25]:
import numpy as np
import torch
import os

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision import models

import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd



# GPU oder CPU auswählen

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("GPU wird benutzt.")
else:
    device = torch.device("cpu")
    print("CPU wird benutzt.")

print("Device:", device)

jubaea_directory = "../Data/Jubaea"
not_jubaea_directory = "../Data/Not_Jubaea"
img_format = (".jpg", ".jpeg", ".png", ".webp")

#Erstelle Klassennamen aus den Ordnern
class_names = ["Jubaea Chilensis"] +sorted(
    d for d in os.listdir(not_jubaea_directory)
    if os.path.isdir(os.path.join(not_jubaea_directory,d))    #Übergibt nur Namen der Ordner,nicht der Datein
)

#Klassennamen Zahlen zuweisen
class_enumerate = {name :i for i, name in enumerate(class_names)}
class_id = len(class_enumerate)


#  Daten importieren
def create_sample():
    sample = []

    for image in os.listdir(jubaea_directory):
        if image.lower().endswith((img_format)):
            sample.append(
                (os.path.join(jubaea_directory, image), class_enumerate["Jubaea Chilensis"])
            )


    #Iteriere für alle anderen Palmenarten durch
    for name in class_names:

        folder_path = os.path.join(
            not_jubaea_directory,
            name
        )

        if not os.path.isdir(folder_path):
            continue

        for image in os.listdir(folder_path):

            if image.lower().endswith((img_format)):
                sample.append((os.path.join(folder_path,image),class_enumerate[name]))

    return sample


sample = create_sample()

pictures = [x[0] for x in sample]
labels = [x[1] for x in sample]

#Daten cappen
data_fraction = 0.7       #Anpassen
if data_fraction < 1:
    pictures,_, labels,_ = train_test_split(
        pictures,
        labels,
        train_size=data_fraction,
        stratify=labels,
        random_state=42
    )


#Neuer Trainingsplit mit Validation 70/15/15

#Hier 70% Data zum trainieren
X_train, X_temp, Y_train, Y_temp = train_test_split(
    pictures,
    labels,
    test_size=0.3,
    stratify=labels,
    random_state=42
)

# 15/15 Val und Testing
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp,
    Y_temp,
    test_size=0.5,
    stratify=Y_temp,
    random_state=42
)

print("Training: ",len(X_train))
print("Validation: ",len(X_val))
print("Testing: ",len(X_test))


# Transformationen


# Werte, die beim ursprünglichen Training von ResNet18 auf ImageNet benutzt wurden
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]


# Resize + Data Augmentation + Normalisierung
train_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.RandomRotation(15),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])


test_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])



# Dataset


class JubaeaDataset(Dataset):

    def __init__(
        self,
        paths,
        labels,
        transform=None
    ):

        self.paths = paths
        self.labels = labels
        self.transform = transform


    def __len__(self):
        return len(self.paths)


    def __getitem__(self, index):

        path = self.paths[index]

        label = self.labels[index]

        image = Image.open(path).convert("RGB")

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        if self.transform:
            image = self.transform(image)

        return image, label


train_dataset = JubaeaDataset(
    X_train,
    Y_train,
    transform=train_transform
)

val_dataset = JubaeaDataset(
    X_val,
    Y_val,
    transform=test_transform
)

test_dataset = JubaeaDataset(
    X_test,
    Y_test,
    transform=test_transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)


# ResNet18 laden
net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
net.fc = nn.Linear(net.fc.in_features, class_id)

"""
# Bereits trainiertes Jubaea-Modell laden
###net.load_state_dict(
###    torch.load(
###        "Jubaea_resnet.pth",
###        map_location=device
###   )
###)
"""

# GPU benutzen
net = net.to(device)

# Lossfunktion
loss_fn = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(
    net.parameters(),
    lr=0.0001,
    weight_decay=5e-4
)

# Training
epochs = 10

best_val_acc = 0

classes = class_names

for epoch in range(epochs):

    net.train()

    loss_calc = 0.0

    correct_train = 0
    total_train = 0


    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)


        optimizer.zero_grad()       # Alte Gradienten löschen
        outputs = net(images)

        loss = loss_fn(             # Loss berechnen
            outputs,
            labels
        )


        loss.backward()             #Backpropagation

        # Gewichte verändern
        optimizer.step()

        loss_calc += loss.item()


        # Trainingsaccuracy berechnen

        preds = torch.argmax(
            outputs,
            dim=1
        )

        correct_train += (
            preds == labels
        ).sum().item()

        total_train += labels.size(0)


    epoch_loss = (
        loss_calc /
        len(train_loader)
    )

    train_acc = (
        correct_train /
        total_train
    )


    print()
    print(
        f"Epoch {epoch + 1}/{epochs}"
    )

    print(
        "Trainingsloss:",
        epoch_loss
    )

    print(
        "Trainingsaccuracy:",
        train_acc
    )


    # Validation
    net.eval()

    correct_val = 0
    total_val = 0


    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)      #Testdaten auf GPU
            labels = labels.to(device)

            outputs = net(images)

            probs = torch.softmax(
                outputs,
                dim=1
            )

            preds = torch.argmax(
                outputs,
                dim=1
            )

            correct_val+= (
                preds == labels
            ).sum().item()

            total_val += labels.size(0)

        val_acc = correct_val / total_val
        print("Validationsaccuracy:", val_acc)



    # Bestes Modell speichern

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            net.state_dict(),
            "../multiclass_ver1_5.pth"
        )

        print("Neues bestes Modell gespeichert.")



GPU wird benutzt.
Device: mps
Training:  5910
Validation:  1266
Testing:  1267

Epoch 1/10
Trainingsloss: 1.4234198735606285
Trainingsaccuracy: 0.5302876480541455
Validationsaccuracy: 0.6848341232227488
Neues bestes Modell gespeichert.

Epoch 2/10
Trainingsloss: 0.7313866035271717
Trainingsaccuracy: 0.7543147208121828
Validationsaccuracy: 0.7488151658767772
Neues bestes Modell gespeichert.

Epoch 3/10
Trainingsloss: 0.5281890329494271
Trainingsaccuracy: 0.8230118443316413
Validationsaccuracy: 0.7456556082148499

Epoch 4/10
Trainingsloss: 0.3831260106576386
Trainingsaccuracy: 0.8734348561759729
Validationsaccuracy: 0.7598736176935229
Neues bestes Modell gespeichert.

Epoch 5/10
Trainingsloss: 0.29739037636787663
Trainingsaccuracy: 0.898984771573604
Validationsaccuracy: 0.7456556082148499

Epoch 6/10
Trainingsloss: 0.22126556836789654
Trainingsaccuracy: 0.9267343485617597
Validationsaccuracy: 0.7661927330173776
Neues bestes Modell gespeichert.

Epoch 7/10
Trainingsloss: 0.176740952717360

In [30]:

#Das beste Model laden
net.load_state_dict(
    torch.load(
        "../multiclass_ver1_5.pth",
        map_location=device
    )
)


net = net.to(device)
net.eval()

all_preds = []
all_labels = []
# Test
correct_test = 0
total_test = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = net(images)

        preds = torch.argmax(outputs,dim=1)
        correct_test += (preds == labels).sum().item()

        total_test += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

cmatrix = confusion_matrix(
    all_labels,
    all_preds
)
test_acc = correct_test / total_test
print("Acc nach Test: ", test_acc)

#Tabelle erstellen
cm_df = pd.DataFrame(cmatrix,classes,classes)
print(cm_df)

#Precicion, Recall, F1
report = classification_report(
    y_true=all_labels,
    y_pred=all_preds,
target_names=class_names,
digits=3)
print(report)

# Zeigen von Fehlklassifikationen, bei mehreren Epochen ausschalten
for i in range(len(images)):
    probs = torch.softmax(net(images),dim=1)

    if preds[i] != labels[i]:
        plt.imshow(images[i].detach().cpu().permute(1, 2, 0))
        plt.title(
            f"Falsch\n"
            f"Zugehörige Klasse: {classes[labels[i].item()]}\n"
            f"Vorhersage: {classes[preds[i].item()]} |\n "
            f"P({classes[preds[i].item()]})={probs[i][preds[i]].item():.3}\n"
        )
        plt.axis('off')
        plt.show()



KeyboardInterrupt: 